In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1. Setup Data
corpus = [
    "the king loves the queen",
    "the queen loves the king",
    "the dwarf hates the king",
    "the knight protects the queen",
    "the poison kills the dwarf"
]

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(corpus)
vocab_size = len(tokenizer.word_index) + 1
words = list(tokenizer.word_index.keys())

def plot_embeddings(model, title):
    # Find the first Embedding layer in the model
    embedding_layer = None
    for layer in model.layers:
        if isinstance(layer, layers.Embedding):
            embedding_layer = layer
            break

    if embedding_layer is None:
        print(f"Warning: No Embedding layer found in model '{title}'. Cannot plot embeddings.")
        return

    weights = embedding_layer.get_weights()[0]
    tsne = TSNE(n_components=2, perplexity=min(5, len(words)-1), init='pca', random_state=42)
    vectors_2d = tsne.fit_transform(weights[1:vocab_size]) # Skip padding index 0

    plt.figure(figsize=(6, 4))
    for i, word in enumerate(words):
        plt.scatter(vectors_2d[i, 0], vectors_2d[i, 1])
        plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]))
    plt.title(title)
    plt.grid(True)
    plt.show()

# --- 2. CBOW Implementation ---
X_cbow, y_cbow = [], []
sequences = tokenizer.texts_to_sequences(corpus)
for seq in sequences:
    for i in range(1, len(seq) - 1):
        X_cbow.append([seq[i-1], seq[i+1]]) # Context: 1-left, 1-right
        y_cbow.append(seq[i])
X_cbow, y_cbow = np.array(X_cbow), np.array(y_cbow)

cbow_in = layers.Input(shape=(2,))
cbow_emb = layers.Embedding(vocab_size, 10)(cbow_in)
cbow_avg = layers.GlobalAveragePooling1D()(cbow_emb)
cbow_out = layers.Dense(vocab_size, activation='softmax')(cbow_avg)
cbow_model = Model(cbow_in, cbow_out)
cbow_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
cbow_model.fit(X_cbow, y_cbow, epochs=300, verbose=0)
plot_embeddings(cbow_model, "CBOW Clusters")

# --- 3. Skip-Gram Implementation ---
# Using the built-in skipgrams generator for positive/negative samples
pairs, labels = [], []
for seq in sequences:
    # Added an integer seed to bypass the internal float-to-int conversion bug
    p, l = tf.keras.preprocessing.sequence.skipgrams(seq, vocab_size, window_size=1, seed=42)
    pairs.extend(p)
    labels.extend(l)
pairs, labels = np.array(pairs), np.array(labels)

target_in = layers.Input(shape=(1,))
context_in = layers.Input(shape=(1,))
sg_emb = layers.Embedding(vocab_size, 10)
t_vec = layers.Flatten()(sg_emb(target_in))
c_vec = layers.Flatten()(sg_emb(context_in))
dot = layers.Dot(axes=1)([t_vec, c_vec])
sg_out = layers.Dense(1, activation='sigmoid')(dot)
sg_model = Model([target_in, context_in], sg_out)
sg_model.compile(optimizer='adam', loss='binary_crossentropy')
sg_model.fit([pairs[:, 0], pairs[:, 1]], labels, epochs=300, verbose=0)
plot_embeddings(sg_model, "Skip-Gram Clusters")

# --- 4. Skip-Thought (Seq2Seq) Implementation ---
# Predict Sentence(i+1) from Sentence(i)
input_texts = sequences[:-1]
target_texts = sequences[1:]
# Pad for uniform shape
input_texts = pad_sequences(input_texts, padding='post')
target_texts = pad_sequences(target_texts, padding='post')

enc_in = layers.Input(shape=(None,))
enc_emb = layers.Embedding(vocab_size, 10)(enc_in)
_, state = layers.GRU(16, return_state=True)(enc_emb)

dec_in = layers.Input(shape=(None,))
dec_emb = layers.Embedding(vocab_size, 10)(dec_in)
dec_gru = layers.GRU(16, return_sequences=True)(dec_emb, initial_state=state)
dec_out = layers.Dense(vocab_size, activation='softmax')(dec_gru)

ss_model = Model([enc_in, dec_in], dec_out)
ss_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
# Train: Using input sentence to predict target sentence
ss_model.fit([input_texts, target_texts], target_texts, epochs=300, verbose=0)
plot_embeddings(ss_model, "Skip-Thought Clusters")